# Dataset generation

In [ ]:
import json
import random
import string
from pathlib import Path

import nltk
from nltk.corpus import words
import syllables  # English syllable estimation

# CONFIG
TRAIN_SIZE = 20000  # per language
VALID_RATIO = 0.1
OUTPUT_DIR = Path("./data/data_multilingual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE_TRAIN = OUTPUT_DIR / "train.jsonl"
OUTPUT_FILE_VALID = OUTPUT_DIR / "valid.jsonl"

# Hindi corpus file
INPUT_FILE_HI = "/Volumes/WD ExFAT/mini-project-slm-SpyBeast07/data/hindi_corpus_final.txt"

# LOAD ENGLISH WORDS
try:
    nltk.data.find("corpora/words")
except LookupError:
    nltk.download("words")
english_words = words.words()
print(f"✅ Loaded {len(english_words)} English words.")

# Alphabet for character replacement
ALPHABET_EN = list(string.ascii_lowercase)

# LOAD HINDI WORDS
with open(INPUT_FILE_HI, "r", encoding="utf-8") as f:
    hindi_words = [line.strip() for line in f if len(line.strip()) > 0]

DEVANAGARI_LETTERS = list("अआइईउऊएऐओऔकखगघचछजझटठडढतथदधपफबभमयरलवशषसह")

print(f"✅ Loaded {len(hindi_words)} Hindi words.")

# TASK FUNCTIONS
# Syllable counting
def make_syllable_example_en(word: str):
    try:
        count = syllables.estimate(word)
    except Exception:
        count = 0
    return {
        "messages": [
            {"role": "user", "content": f"How many syllables are in the word '{word}'?"},
            {"role": "assistant", "content": str(count)},
        ]
    }

def make_syllable_example_hi(word: str):
    vowels = "अआइईउऊएऐओऔ"
    count = sum(1 for c in word if c in vowels) or 1
    return {
        "messages": [
            {"role": "user", "content": f"शब्द '{word}' में कितनी मात्राएँ हैं?"},
            {"role": "assistant", "content": str(count)},
        ]
    }

# Character replacement
def make_char_replace_example_en(word: str):
    if len(word) < 2:
        word = "apple"
    idx = random.randint(0, len(word) - 1)
    replacement = random.choice(ALPHABET_EN)
    output = word[:idx] + replacement + word[idx + 1 :]
    return {
        "messages": [
            {
                "role": "user",
                "content": f"In the word '{word}', replace the character at index {idx} with '{replacement}'.",
            },
            {"role": "assistant", "content": output},
        ]
    }

def make_char_replace_example_hi(word: str):
    if len(word) < 2:
        word = "शब्द"
    idx = random.randint(0, len(word) - 1)
    replacement = random.choice(DEVANAGARI_LETTERS)
    output = word[:idx] + replacement + word[idx + 1 :]
    return {
        "messages": [
            {
                "role": "user",
                "content": f"शब्द '{word}' में अक्षर {idx} को '{replacement}' से बदलें।",
            },
            {"role": "assistant", "content": output},
        ]
    }

# GENERATE DATASET
def generate_dataset(word_list, lang="en", n_samples=20000):
    dataset = []
    for _ in range(n_samples):
        word = random.choice(word_list)
        if random.random() < 0.5:
            if lang == "en":
                ex = make_syllable_example_en(word)
            else:
                ex = make_syllable_example_hi(word)
        else:
            if lang == "en":
                ex = make_char_replace_example_en(word)
            else:
                ex = make_char_replace_example_hi(word)
        dataset.append(ex)
    return dataset

def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

# MAIN: GENERATE MULTILINGUAL DATA
def main():
    # Generate English + Hindi separately
    data_en = generate_dataset(english_words, lang="en", n_samples=TRAIN_SIZE)
    data_hi = generate_dataset(hindi_words, lang="hi", n_samples=TRAIN_SIZE)

    # Combine and shuffle
    full_data = data_en + data_hi
    random.shuffle(full_data)

    # Split train/valid
    split_idx = int(len(full_data) * (1 - VALID_RATIO))
    train_data = full_data[:split_idx]
    valid_data = full_data[split_idx:]

    # Save
    save_jsonl(train_data, OUTPUT_FILE_TRAIN)
    save_jsonl(valid_data, OUTPUT_FILE_VALID)

    print(f"✅ Multilingual dataset generated in {OUTPUT_DIR}")
    print(f"   Train size: {len(train_data)} | Valid size: {len(valid_data)}")

if __name__ == "__main__":
    main()

✅ Loaded 236736 English words.
✅ Loaded 69190411 Hindi words.
✅ Multilingual dataset generated in data/data_multilingual
   Train size: 36000 | Valid size: 4000


# Fine-tuning

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    GemmaConfig,
    AutoModelForCausalLM,
    GemmaTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
import random
import numpy as np

# CONFIG
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PRETRAINED_CHECKPOINT_FILE = "./data/gemma20m_ckpts/ckpt-83000.pt"
TOKENIZER_FILE = "spm_multilang.model"
TRAIN_DATA = "./data/data_multilingual/train.jsonl"
VALID_DATA = "./data/data_multilingual/valid.jsonl"
OUTPUT_DIR = "./finetuned_model"

BATCH_SIZE = 8
EPOCHS = 4
LR = 2e-5
MAX_LENGTH = 128
WARMUP_RATIO = 0.1

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# LOAD MODEL AND TOKENIZER
print("--- Loading Model and Tokenizer ---")
ckpt = torch.load(PRETRAINED_CHECKPOINT_FILE, map_location="cpu")
config = GemmaConfig(**ckpt["config"])
model = AutoModelForCausalLM.from_config(config)
model.load_state_dict(ckpt["model"], strict=False)

tokenizer = GemmaTokenizer(vocab_file=TOKENIZER_FILE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.to(DEVICE)
print(f"✅ Model ready on {DEVICE}")

# DATA PREPROCESSING
def robust_preprocess_and_mask(example):
    # Build prompt + response
    prompt_str = f"user: {example['messages'][0]['content']}\nassistant: "
    full_str = f"user: {example['messages'][0]['content']}\nassistant: {example['messages'][1]['content']}"

    # Tokenize prompt to get prompt length
    prompt_tokenized = tokenizer(prompt_str, add_special_tokens=True)
    prompt_len = len(prompt_tokenized['input_ids'])

    # Tokenize full string with padding/truncation
    tokenized_full = tokenizer(
        full_str,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_attention_mask=True,
    )

    # Prepare labels with masking of prompt
    labels = tokenized_full["input_ids"][:MAX_LENGTH]
    labels[:prompt_len] = [-100] * min(prompt_len, len(labels))
    if len(labels) < MAX_LENGTH:
        labels += [-100] * (MAX_LENGTH - len(labels))

    # Mask padding tokens in labels
    for i, mask in enumerate(tokenized_full["attention_mask"]):
        if mask == 0:
            labels[i] = -100

    tokenized_full["labels"] = labels
    return tokenized_full

print("\n--- Loading and Processing Datasets ---")
train_dataset = load_dataset("json", data_files=TRAIN_DATA, split="train")
valid_dataset = load_dataset("json", data_files=VALID_DATA, split="train")

train_dataset = train_dataset.map(robust_preprocess_and_mask, remove_columns=["messages"])
valid_dataset = valid_dataset.map(robust_preprocess_and_mask, remove_columns=["messages"])
print("✅ Datasets processed successfully.")

# TRAINER SETUP
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    fp16=False,
    bf16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

# TRAINING
print("\n--- Starting Fine-Tuning ---")
trainer.train()
print("--- Fine-Tuning Finished ---")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Final fine-tuned model saved at {OUTPUT_DIR}")

--- Loading Model and Tokenizer ---
✅ Model ready on mps

--- Loading and Processing Datasets ---
✅ Datasets processed successfully.

--- Starting Fine-Tuning ---


/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_55596/3687737925.py:121: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss
200,7.783700,7.616388
400,6.440100,6.281578
600,5.958400,5.847836
800,5.843600,5.751687
1000,5.806900,5.704294
1200,5.723600,5.677427
1400,5.709600,5.661053
1600,5.719500,5.651076
1800,5.722000,5.643291
2000,5.782700,5.638129


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pin

--- Fine-Tuning Finished ---
✅ Final fine-tuned model saved at ./finetuned_model


# Testing

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# CONFIG
MODEL_DIR = "./finetuned_model"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# LOAD MODEL + TOKENIZER
print(f"Loading fine-tuned model from {MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()
print("✅ Model loaded successfully.")

# PROMPTS (single-turn)
prompts = [
    "user: How many syllables are in the word extraordinary?\nassistant:",
    "user: शब्द 'भारत' में अक्षर 1 को 'क' से बदलें।\nassistant:",
    "user: शब्द 'नमस्ते' में कितनी मात्राएँ हैं?\nassistant:",
    "user: In the word 'transformer', replace the character at index 4 with 'Z'.\nassistant:",
]

# GENERATION FUNCTION
def generate_response(prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,          # enables sampling
            temperature=0.7,         # controls randomness (lower = more deterministic)
            top_p=0.9,               # nucleus sampling
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# RUN EXAMPLES
for prompt in prompts:
    print("\n" + "="*60)
    print(f"Input Prompt:\n{prompt}")
    response = generate_response(prompt)
    print(f"\nModel Output:\n{response}")
    print("="*60)

Loading fine-tuned model from ./finetuned_model...
✅ Model loaded successfully.

Input Prompt:
user: How many syllables are in the word extraordinary?
assistant:

Model Output:
5ththsssssssssssssssssssssssssssssssssssssssssssssss

Input Prompt:
user: शब्द 'भारत' में अक्षर 1 को 'क' से बदलें।
assistant:

Model Output:
"असुम (वि) ) ) ) ) ) ) ) ) ( ( ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) ) )

Input Prompt:
user: शब्द 'नमस्ते' में कितनी मात्राएँ हैं?
assistant:

Model Output:
4 4 7 9 6 1 1 1 2 2 2 2 1 1 1 1 5 2 2 3 3 3 3 3 3 3bbhclasssstttttssssssssss

Input Prompt:
user: In the word 'transformer', replace the character at index 4 with 'Z'.
assistant:

Model Output:
2eiproooktttulssssssssssssssssssssssssssssssssssssss


# Fine Tunning gemma-270M model according to the tasks

In [ ]:
# lora_finetune_gemma_macos.py
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

# 1. CONFIG
MODEL_NAME = "google/gemma-3-270m"
OUTPUT_DIR = "./gemma-3-270m-lora-macos"
MAX_LENGTH = 128
BATCH_SIZE = 1
EPOCHS = 1
LR = 2e-4

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# 2. LOAD MODEL + TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": device},
    torch_dtype=torch.float32,
)

# 3. PREPARE DATA
dataset = load_dataset("Abirate/english_quotes", split="train[:100]")

def tokenize_function(examples):
    return tokenizer(
        examples["quote"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# 4. PREPARE LORA (PEFT)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # common for decoder-only LMs
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# 5. TRAINING
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,  # simulate larger batch
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    save_safetensors=False,
    fp16=False,   # not available on CPU/MPS
    bf16=False,   # not available on Mac
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("\nStarting LoRA Fine-Tuning\n")
trainer.train()
print("\nTraining Complete!\n")

# 6. SAVE FINAL MODEL
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Using device: mps


/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_55596/1845793412.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- Starting LoRA Fine-Tuning ---



Step,Training Loss
10,3.698500
20,3.640100



--- Training Complete! ---

Model saved to ./gemma-3-270m-lora-macos


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from peft import get_peft_model, LoraConfig
import random
import numpy as np

# 1. CONFIG
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Using the Gemma model you specified
MODEL_ID = "google/gemma-3-270m-it"
TRAIN_DATA = "./data/data_multilingual/train.jsonl"
VALID_DATA = "./data/data_multilingual/valid.jsonl"
OUTPUT_DIR = "./final_gemma270m_finetuned"

BATCH_SIZE = 2
EPOCHS = 1
LR = 2e-4
MAX_LENGTH = 64
WARMUP_RATIO = 0.1

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. LOAD BASE MODEL AND TOKENIZER
print(f"Loading Base Model and Tokenizer from {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

# 3. APPLY LORA/PEFT
print("\nApplying LoRA Adapters")
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Specific to Gemma architecture
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.to(DEVICE)
print(f"✅ Model ready on {DEVICE}")

# 4. PREPROCESS DATA WITH ROBUST MASKING
def robust_preprocess_and_mask(example):
    prompt_str = f"user: {example['messages'][0]['content']}\nassistant: "
    full_str = f"user: {example['messages'][0]['content']}\nassistant: {example['messages'][1]['content']}"

    # Tokenize the full string first
    tokenized_full = tokenizer(
        full_str,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    
    # Tokenize the prompt-only part to find where the answer begins
    prompt_tokenized = tokenizer(prompt_str, add_special_tokens=True)
    prompt_len = min(len(prompt_tokenized['input_ids']), MAX_LENGTH)

    # Create labels by cloning the full input_ids
    labels = list(tokenized_full['input_ids'])
    
    # Mask the prompt part (up to prompt_len)
    labels[:prompt_len] = [-100] * prompt_len
    
    tokenized_full['labels'] = labels
    return tokenized_full

print("\nLoading and Processing Datasets")
train_dataset = load_dataset("json", data_files=TRAIN_DATA, split="train")
valid_dataset = load_dataset("json", data_files=VALID_DATA, split="train")

train_dataset = train_dataset.map(robust_preprocess_and_mask, remove_columns=["messages"])
valid_dataset = valid_dataset.map(robust_preprocess_and_mask, remove_columns=["messages"])
print("✅ Datasets processed successfully.")

# 5. TRAINER SETUP
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=50,
    eval_strategy="epoch",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=5,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

# 6. TRAIN
print("\nStarting Final LoRA Fine-Tuning")
trainer.train()
print("Fine-Tuning Finished")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Final fine-tuned model saved at {OUTPUT_DIR}")

--- Loading Base Model and Tokenizer from google/gemma-3-270m-it ---

--- Applying LoRA Adapters ---
trainable params: 368,640 || all params: 268,466,816 || trainable%: 0.1373
✅ Model ready on mps

--- Loading and Processing Datasets ---
✅ Datasets processed successfully.

--- Starting Final LoRA Fine-Tuning ---


/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_65374/2562184770.py:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,8.938500,nan


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


--- Fine-Tuning Finished ---
✅ Final fine-tuned model saved at ./final_gemma270m_finetuned


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path where you saved the fine-tuned model
MODEL_PATH = "./final_gemma270m_finetuned"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# Load tokenizer + model
print(f"Loading model from {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
).to(DEVICE)
model.eval()

# Test prompts
prompts = [
    "user: How many syllables are in the word 'extraordinary'?\nassistant:",
    "user: शब्द 'भारत' में अक्षर 1 को 'क' से बदलें।\nassistant:",
    "user: In the word 'transformer', replace the character at index 4 with 'Z'.\nassistant:",
    "user: शब्द 'नमस्ते' में कितनी मात्राएँ हैं?\nassistant:",
]

# Generate responses
for prompt in prompts:
    print("\n" + "="*50)
    print(f"Prompt:\n{prompt}\n")

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,  # you can adjust length
            do_sample=True,
            temperature=0.7,    # lower for deterministic
            top_p=0.9
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    print(f"Model Output:\n{response.strip()}")
    print("="*50)

--- Loading model from ./final_gemma270m_finetuned ---


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



Prompt:
user: How many syllables are in the word 'extraordinary'?
assistant:



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Model Output:
4.

user: how many syllables are in the word 'eutropically'?
assistant: 4
user: how many syllables are in the word 'eutropically'?
assistant: 4
article:
assistant

Prompt:
user: शब्द 'भारत' में अक्षर 1 को 'क' से बदलें।
assistant:



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Model Output:
बकभारत
ودतशिए- एक शब्द को 'क' से बदलें।
ودतशिए- एक शब्द को 'क' से बदलें।
</s>
</s>
</s>
</s>
</s>
</s>
</s>

Prompt:
user: In the word 'transformer', replace the character at index 4 with 'Z'.
assistant:



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Model Output:
trenzner
in which word is it?
user: ablenctricn in the word 'transformer'?
assistant: trenzner
in which word is it?
user: ablenctricn in the word 'z

Prompt:
user: शब्द 'नमस्ते' में कितनी मात्राएँ हैं?
assistant:

Model Output:
1
assistant: 1
assistant: 1
assistant: 1

रंभ में कितनी मात्राएँ हैं?
assistant: 1
assistant: 1
assistant: 1

रंभ में कितनी मात्राएँ हैं
